# [STARTER] Exercise - Build a Web-Aware Agent with Search and Knowledge Comparison

In this exercise, you'll build an agent that can search the web for current information and compare
it with its internal knowledge. This demonstrates how to enhance an LLM's capabilities with real-time
web data and how to critically analyze differences between sources.


## Challenge

Your task is to create an agent that can:

- Implement web search functionality using Tavily API
- Parse and process search results effectively
- Handle different types of queries (news, facts, events)
- Extract relevant information from search results

## Setup
First, let's import the necessary libraries:

In [1]:
import os
from datetime import datetime
from typing import List, Dict
from dotenv import load_dotenv
from tavily import TavilyClient

from lib.agents import Agent
from lib.messages import BaseMessage
from lib.tooling import tool

In [2]:
load_dotenv()

True

## Play with Tavily

In [3]:
api_key = os.getenv("TAVILY_API_KEY")
client = TavilyClient(api_key=api_key)

In [4]:
# [TODO] Define a query and run
query = "Who won the 2026 Paulista Championships in Brazil?"
result = client.search(query)

In [5]:
result

{'query': 'Who won the 2026 Paulista Championships in Brazil?',
 'response_time': 0.79,
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://www.youtube.com/watch?v=WqUeCYpIHZk',
   'title': 'Palmeiras wins the first leg of the 2026 Paulista Championship final.',
   'content': 'Palmeiras wins the first leg of the 2026 Paulista Championship final. ... ✈️ GRINGA NO BRASIL: ASSISTINDO JOGO NO ALLIANZ PARQUE ft IMAN.',
   'score': 0.99966204,
   'raw_content': None},
  {'url': 'https://www.transferhunt.com/competition/season/2026/brazilian-campeonato-paulista-a1-341402/results',
   'title': 'Brazilian Campeonato Paulista A1 - all results for the season 26/27',
   'content': 'Country: Brazilian Campeonato Paulista A1 country flag. Brazil · Season: 2026 2025 2024 · Title holder: Palmeiras (26) · Most titles: Corinthians Paulista (SP) (',
   'score': 0.99828595,
   'raw_content': None},
  {'url': 'https://www.worldfootball.net/competition/co4521/brazil-c

## Define Web Search tool

In [6]:
# [TODO] Define the search tool
@tool
def web_search(query: str) -> Dict:
    """"
    Perform a web search using the Tavily API and return the results.
     Args:
        query (str): The search query.
    Returns:
        Dict: The search results from the Tavily API.
    """
    api_key = os.getenv("TAVILY_API_KEY")
    client = TavilyClient(api_key=api_key)

    search_result = client.search(
                query=query,
                search_depth="advanced",
                include_summary=True,
                include_raw_content=False,
                include_images=False)
    
    formatted_result = {
        "answer": search_result.get("answer", ""),
        "results": search_result.get("results", []),
        "search_metadata": {
            "timestamp": datetime.now().isoformat(),
            "query": query,
        }
    }

    return formatted_result

In [7]:
tools = [web_search]

## Define Agents

The first agent should not use tools, just its own knowledge. The second one should have web search tool enabled.

In [8]:
# [TODO] Agent with no tools
simple_agent = Agent(
    model_name="gpt-5-nano",
    instructions="You are an assistant that can help with general questions about brazilian soccer.",
)

In [9]:
# [TODO] Agent with web search tool
web_agent = Agent(
    model_name="gpt-5-nano",
    instructions="You are an assistant that can help with general questions about brazilian soccer.",
    tools=tools
)

In [10]:
def print_messages(messages: List[BaseMessage]):
    for m in messages:
        print(f" -> (role = {m.role}, content = {m.content}, tool_calls = {getattr(m, 'tool_calls', None)})")

## Run your Agents

Run the Agents and compare them. The following queries are just for reference. Change them as you want.

**Simple Agent**

**Note**: This example relies on the date being recent enough that the answer will not be in the model's training data. Try with other current events/dates if needed to get similar results.

In [11]:
run1 = simple_agent.invoke(
    query="Who won the 2026 Paulista Championships in Brazil?", 
)

print("\nMessages from run 1:")
messages = run1.get_final_state()["messages"]
print_messages(messages)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__

Messages from run 1:
 -> (role = system, content = You are an assistant that can help with general questions about brazilian soccer., tool_calls = None)
 -> (role = user, content = Who won the 2026 Paulista Championships in Brazil?, tool_calls = None)
 -> (role = assistant, content = Não tenho acesso a resultados em tempo real. Meu conhecimento vai até 2024, então não posso confirmar quem venceu o Paulista de 2026.

Se quiser, posso ajudar de outras formas:
- explicar o formato atual do Campeonato Paulista (fases, fase de grupos, mata-mata, etc.)
- listar os campeões históricos até 2024 (com fontes para você verificar)
- indicar onde checar o campeão de 2026 em fontes oficiais (Federação Paulista de Futebol, Globo Esporte, Wikipedia)

Qual opção você prefere? Também posso procurar se você me fornecer um link ou me indic

In [12]:
print(run1.get_final_state()["messages"][-1].content)

Não tenho acesso a resultados em tempo real. Meu conhecimento vai até 2024, então não posso confirmar quem venceu o Paulista de 2026.

Se quiser, posso ajudar de outras formas:
- explicar o formato atual do Campeonato Paulista (fases, fase de grupos, mata-mata, etc.)
- listar os campeões históricos até 2024 (com fontes para você verificar)
- indicar onde checar o campeão de 2026 em fontes oficiais (Federação Paulista de Futebol, Globo Esporte, Wikipedia)

Qual opção você prefere? Também posso procurar se você me fornecer um link ou me indicar uma fonte para resumir.


In [20]:
run2 = simple_agent.invoke(
    query="Which war is the last occurred?", 
)
print("\nMessages from run 2:")
messages = run2.get_final_state()["messages"]
print_messages(messages)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__

Messages from run 2:
 -> (role = system, content = You are an assistant that can help with general questions about brazilian soccer., tool_calls = None)
 -> (role = user, content = Who won the 2026 Paulista Championships in Brazil?, tool_calls = None)
 -> (role = assistant, content = Não tenho acesso a resultados em tempo real. Meu conhecimento vai até 2024, então não posso confirmar quem venceu o Paulista de 2026.

Se quiser, posso ajudar de outras formas:
- explicar o formato atual do Campeonato Paulista (fases, fase de grupos, mata-mata, etc.)
- listar os campeões históricos até 2024 (com fontes para você verificar)
- indicar onde checar o campeão de 2026 em fontes oficiais (Federação Paulista de Futebol, Globo Esporte, Wikipedia)

Qual opção você prefere? Também posso procurar se você me fornecer um link ou me indic

In [21]:
print(run2.get_final_state()["messages"][-1].content)

Do you mean the most recent war that began, or the most recent one still ongoing, or a war in a specific country/region?

If you just want a quick overview (as of 2024), there isn’t a single “last war.” Several major conflicts have been ongoing or recently started:

- Israel–Hamas war: began October 7, 2023; ongoing.
- Russia–Ukraine war: escalated into a full-scale invasion on February 24, 2022; ongoing (building on a conflict that started in 2014).
- Sudan conflict: intensified in 2023 and has continued with various phases and fronts.
- Myanmar civil war: ongoing since 2021 after the coup and subsequent fighting.

If you specify a region or country, I can give a more precise answer or note the most recently started conflict there.


**Web Agent**

In [15]:
run1 = web_agent.invoke(
    query="Who won the 2026 Paulista Championships in Brazil?", 
)

print("\nMessages from run 1:")
messages = run1.get_final_state()["messages"]
print_messages(messages)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__

Messages from run 1:
 -> (role = system, content = You are an assistant that can help with general questions about brazilian soccer., tool_calls = None)
 -> (role = user, content = Who won the 2026 Paulista Championships in Brazil?, tool_calls = None)
 -> (role = assistant, content = None, tool_calls = [ChatCompletionMessageToolCall(id='call_Ac8pa5CbFNQQHue5pFBaZ8Lj', function=Function(arguments='{"query":"Campeonato Paulista 2026 campeão"}', name='web_search'), type='function')])
 -> (role = tool, content = "{'answer': None, 'results': [{'url': 'https://ge.globo.com/futebol/times/palmeiras/noticia/2026/03/09/titulo-paulista-consagra-nova-geracao-de-campeoes-no-palmeiras-e-aumenta-expectativa-para-2026.ghtml', 'title': 'Palmeiras c

In [16]:
print(run1.get_final_state()["messages"][-1].content)

Palmeiras won the 2026 Campeonato Paulista (Paulistão). It was their 27th Paulista title, beating Novorizontino in the final (1-0 in Barueri and 2-1 in Novo Horizonte; 3-1 on aggregate).


In [19]:
run2 = web_agent.invoke(
    query="Which war is the last occurred?", 
)
print("\nMessages from run 2:")
messages = run2.get_final_state()["messages"]
print_messages(messages)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__

Messages from run 2:
 -> (role = system, content = You are an assistant that can help with general questions about brazilian soccer., tool_calls = None)
 -> (role = user, content = Who won the 2026 Paulista Championships in Brazil?, tool_calls = None)
 -> (role = assistant, content = None, tool_calls = [ChatCompletionMessageToolCall(id='call_Ac8pa5CbFNQQHue5pFBaZ8Lj', function=Function(arguments='{"query":"Campeonato Paulista 2026 campeão"}', name='web_search'), type='function')])
 -> (role = tool, content = "{'answer': None, 'results': [{'url': 'https://ge.globo.com/futebol/times/palmeiras/noticia/2026/03/09/titulo-paulista-consagra-nova-geracao-de-campeoes-no-palmeiras-e-aumenta-expectativa-para-2026.ghtml', 'title': 'Palmeiras campe\u00e3o paulista: nova gera\u00e7\u00e3o brilha e foca em 2026 | Ge', 'content': '## R

In [22]:
print(run2.get_final_state()["messages"][-1].content)

Do you mean the most recent war that began, or the most recent one still ongoing, or a war in a specific country/region?

If you just want a quick overview (as of 2024), there isn’t a single “last war.” Several major conflicts have been ongoing or recently started:

- Israel–Hamas war: began October 7, 2023; ongoing.
- Russia–Ukraine war: escalated into a full-scale invasion on February 24, 2022; ongoing (building on a conflict that started in 2014).
- Sudan conflict: intensified in 2023 and has continued with various phases and fronts.
- Myanmar civil war: ongoing since 2021 after the coup and subsequent fighting.

If you specify a region or country, I can give a more precise answer or note the most recently started conflict there.


## Advanced

You can modify `agents.py` to include: 
- a comparison field in the state schema
- a web search step
- a comparison step in the workflow